# Module 8: Sampling Techniques for LLM Output

## Learning Objectives
By the end of this module, you will be able to:
- Understand how LLMs generate text through sampling
- Control output diversity using Temperature, Top-K, and Top-P
- Use Beam Search for higher quality outputs
- Choose the right sampling strategy for different tasks

---

## 1. How LLMs Generate Text

Large Language Models generate text **one token at a time** by predicting probability distributions over the vocabulary.

```
Input: "The capital of France is"
     ↓
   [LLM]
     ↓
Probabilities for next token:
  "Paris"  → 0.85
  "the"    → 0.05
  "a"      → 0.03
  "known"  → 0.02
  ...      → 0.05
```

### The Key Question
**How do we select the next token from this probability distribution?**

This is where sampling strategies come in!

In [ ]:
!uv pip install -q numpy matplotlib openai transformers torch

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

# Simulated vocabulary and probabilities from an LLM
tokens = ['Paris', 'the', 'a', 'known', 'located', 'beautiful', 'famous', 'called', 'one', 'very']
raw_probs = np.array([0.50, 0.12, 0.10, 0.08, 0.07, 0.05, 0.04, 0.02, 0.01, 0.01])

print("Prompt: 'The capital of France is ___'\n")
print("Model's probability distribution:")
for token, prob in zip(tokens, raw_probs):
    bar = '█' * int(prob * 50)
    print(f"  {token:12} {prob:.2f} {bar}")

---

## 2. Greedy Decoding

**Greedy decoding** always selects the token with the highest probability.

### Pros
- Deterministic (same input → same output)
- Fast and simple

### Cons
- Repetitive and boring outputs
- Can get stuck in loops
- Misses creative alternatives

In [ ]:
def greedy_decode(probs, tokens):
    """Always pick the highest probability token"""
    idx = np.argmax(probs)
    return tokens[idx], probs[idx]

# Greedy decoding - always picks "Paris"
print("Greedy Decoding (5 runs):")
for i in range(5):
    token, prob = greedy_decode(raw_probs, tokens)
    print(f"  Run {i+1}: {token} (p={prob:.2f})")

print("\n💡 Notice: Greedy always picks 'Paris' - zero creativity!")

---

## 3. Temperature Sampling

**Temperature** controls the "sharpness" of the probability distribution.

$$P(x_i) = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

Where:
- $z_i$ = raw logits from the model
- $T$ = temperature parameter

### Effect of Temperature

| Temperature | Effect | Use Case |
|-------------|--------|----------|
| T → 0 | Approaches greedy (deterministic) | Factual Q&A, code |
| T = 1.0 | Original distribution | Balanced tasks |
| T > 1.0 | Flatter, more random | Creative writing, brainstorming |

In [ ]:
def apply_temperature(logits, temperature):
    """Apply temperature scaling and convert to probabilities"""
    # Avoid division by zero
    if temperature < 0.01:
        temperature = 0.01
    
    scaled = logits / temperature
    # Softmax
    exp_scaled = np.exp(scaled - np.max(scaled))  # Subtract max for numerical stability
    return exp_scaled / exp_scaled.sum()

def sample_with_temperature(logits, tokens, temperature):
    """Sample a token with temperature scaling"""
    probs = apply_temperature(logits, temperature)
    idx = np.random.choice(len(tokens), p=probs)
    return tokens[idx], probs[idx]

# Convert probabilities to logits (approximate)
logits = np.log(raw_probs + 1e-10)

# Visualize temperature effects
temperatures = [0.1, 0.5, 1.0, 2.0]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, temp in zip(axes, temperatures):
    scaled_probs = apply_temperature(logits, temp)
    
    bars = ax.bar(range(len(tokens)), scaled_probs, color='steelblue')
    # Highlight max
    max_idx = np.argmax(scaled_probs)
    bars[max_idx].set_color('coral')
    
    ax.set_title(f'Temperature = {temp}', fontsize=12)
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Probability')
    ax.set_ylim(0, 1.0)

plt.suptitle('Effect of Temperature on Probability Distribution', fontsize=14)
plt.tight_layout()
plt.show()

print("💡 Low temperature → 'Paris' dominates | High temperature → more uniform")

In [ ]:
# Demonstrate sampling at different temperatures
print("Sampling with different temperatures (10 samples each):\n")

for temp in [0.1, 0.5, 1.0, 2.0]:
    samples = [sample_with_temperature(logits, tokens, temp)[0] for _ in range(10)]
    unique = len(set(samples))
    print(f"T={temp}: {samples}")
    print(f"       Unique tokens: {unique}/10\n")

---

## 4. Top-K Sampling

**Top-K** limits sampling to only the K most probable tokens.

```
Original: Paris(0.50), the(0.12), a(0.10), known(0.08), located(0.07), ...

Top-K=3:  Paris(0.69), the(0.17), a(0.14)  ← Renormalized!
          (other tokens have 0 probability)
```

### Benefits
- Prevents choosing very unlikely tokens
- Still allows some diversity within top choices

In [ ]:
def top_k_sampling(probs, tokens, k):
    """Sample from only the top-k tokens"""
    # Get indices of top-k probabilities
    top_k_idx = np.argsort(probs)[-k:]
    
    # Zero out non-top-k probabilities
    filtered_probs = np.zeros_like(probs)
    filtered_probs[top_k_idx] = probs[top_k_idx]
    
    # Renormalize
    filtered_probs = filtered_probs / filtered_probs.sum()
    
    # Sample
    idx = np.random.choice(len(tokens), p=filtered_probs)
    return tokens[idx], filtered_probs

# Visualize Top-K filtering
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
k_values = [1, 3, 5]

for ax, k in zip(axes, k_values):
    _, filtered = top_k_sampling(raw_probs, tokens, k)
    
    colors = ['coral' if p > 0 else 'lightgray' for p in filtered]
    ax.bar(range(len(tokens)), filtered, color=colors)
    ax.set_title(f'Top-K = {k}', fontsize=12)
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Probability')
    ax.set_ylim(0, 1.0)

plt.suptitle('Top-K Sampling', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Sample with different K values
print("Top-K Sampling (10 samples each):\n")

for k in [1, 3, 5, 10]:
    samples = [top_k_sampling(raw_probs, tokens, k)[0] for _ in range(10)]
    unique = len(set(samples))
    print(f"K={k:2d}: {samples}")
    print(f"      Unique tokens: {unique}/10\n")

---

## 5. Top-P (Nucleus) Sampling

**Top-P** (also called Nucleus Sampling) dynamically selects the smallest set of tokens whose cumulative probability exceeds threshold P.

### Why Top-P > Top-K?

Top-K has a fixed number of candidates regardless of the distribution:

```
Scenario A: One token has 0.95 probability
  → Top-K=3 includes 2 unlikely tokens (wasteful)
  → Top-P=0.9 includes only 1 token (efficient)

Scenario B: Probabilities are spread evenly
  → Top-K=3 might miss good candidates
  → Top-P=0.9 includes more tokens (adaptive)
```

In [ ]:
def top_p_sampling(probs, tokens, p):
    """Sample from the smallest set of tokens with cumulative prob >= p"""
    # Sort by probability (descending)
    sorted_indices = np.argsort(probs)[::-1]
    sorted_probs = probs[sorted_indices]
    
    # Find cumulative probability
    cumsum = np.cumsum(sorted_probs)
    
    # Find cutoff (first index where cumsum >= p)
    cutoff_idx = np.searchsorted(cumsum, p) + 1
    
    # Keep only top tokens
    top_indices = sorted_indices[:cutoff_idx]
    
    # Filter and renormalize
    filtered_probs = np.zeros_like(probs)
    filtered_probs[top_indices] = probs[top_indices]
    filtered_probs = filtered_probs / filtered_probs.sum()
    
    # Sample
    idx = np.random.choice(len(tokens), p=filtered_probs)
    return tokens[idx], filtered_probs, cutoff_idx

# Visualize Top-P sampling
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
p_values = [0.5, 0.8, 0.95]

for ax, p in zip(axes, p_values):
    _, filtered, n_tokens = top_p_sampling(raw_probs, tokens, p)
    
    colors = ['coral' if prob > 0 else 'lightgray' for prob in filtered]
    ax.bar(range(len(tokens)), filtered, color=colors)
    ax.set_title(f'Top-P = {p} ({n_tokens} tokens)', fontsize=12)
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Probability')
    ax.set_ylim(0, 1.0)

plt.suptitle('Top-P (Nucleus) Sampling', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Demonstrate adaptive nature of Top-P
print("Top-P Sampling (10 samples each):\n")

for p in [0.5, 0.8, 0.95]:
    results = [top_p_sampling(raw_probs, tokens, p) for _ in range(10)]
    samples = [r[0] for r in results]
    n_tokens = results[0][2]
    unique = len(set(samples))
    print(f"P={p:.2f} ({n_tokens} tokens considered): {samples}")
    print(f"              Unique tokens sampled: {unique}/10\n")

---

## 6. Beam Search

**Beam search** maintains multiple candidate sequences (beams) and picks the overall best sequence.

```
Beam width = 3

Step 1: "The" → ["The cat", "The dog", "The bird"]
Step 2: Each beam generates candidates → Keep top 3 overall
        ["The cat sat", "The cat ran", "The dog barked", ...] → Keep best 3
Step 3: Continue until complete

Return: Highest-probability complete sequence
```

### Pros
- Finds higher-quality sequences than greedy
- Explores alternatives

### Cons
- Slower (maintains multiple hypotheses)
- Can still be repetitive
- Best for structured outputs (translation, summary)

In [ ]:
# Simplified beam search visualization
def visualize_beam_search():
    """Visual representation of beam search process"""
    fig, ax = plt.subplots(figsize=(14, 6))
    
    # Beam search tree (simplified)
    tree = {
        "Start": (0.5, 0.9),
        "The": (0.25, 0.7),
        "A": (0.75, 0.7),
        "The cat": (0.15, 0.5),
        "The dog": (0.35, 0.5),
        "A small": (0.65, 0.5),
        "The cat sat": (0.1, 0.3),
        "The dog ran": (0.3, 0.3),
        "A small bird": (0.6, 0.3),
    }
    
    edges = [
        ("Start", "The"), ("Start", "A"),
        ("The", "The cat"), ("The", "The dog"),
        ("A", "A small"),
        ("The cat", "The cat sat"), ("The dog", "The dog ran"),
        ("A small", "A small bird")
    ]
    
    # Draw edges
    for start, end in edges:
        x1, y1 = tree[start]
        x2, y2 = tree[end]
        ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                   arrowprops=dict(arrowstyle="->", color='gray', lw=1.5))
    
    # Draw nodes
    for label, (x, y) in tree.items():
        color = 'coral' if label in ['The cat sat', 'The dog ran', 'A small bird'] else 'steelblue'
        ax.scatter(x, y, s=200, c=color, zorder=5)
        ax.annotate(label, (x, y), fontsize=9, ha='center', va='bottom', 
                   xytext=(0, 10), textcoords='offset points')
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0.2, 1.0)
    ax.set_title('Beam Search (width=3): Exploring Multiple Paths', fontsize=14)
    ax.axis('off')
    
    plt.tight_layout()
    plt.show()

visualize_beam_search()
print("💡 Beam search keeps the top-k candidates at each step, not just one.")

---

## 7. Comparing Strategies with OpenAI API

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

# --- API Key Setup ---
# Option 1 — Google Colab: Load API key from Colab Secrets
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Option 2 — Local (VSCode / Jupyter): Load API key from .env file
load_dotenv()

client = OpenAI()

def generate_text(prompt, temperature=1.0, top_p=1.0, n=1):
    """Generate text with specified sampling parameters"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        top_p=top_p,
        max_tokens=50,
        n=n
    )
    if n == 1:
        return response.choices[0].message.content
    return [c.message.content for c in response.choices]

print("✅ Ready to compare sampling strategies!")

In [ ]:
# Compare temperature settings
prompt = "Write a creative one-sentence opening for a fantasy novel:"

print(f"Prompt: {prompt}\n")
print("=" * 60)

for temp in [0.0, 0.5, 1.0, 1.5]:
    print(f"\n🌡️ Temperature = {temp}:")
    for i in range(2):
        result = generate_text(prompt, temperature=temp)
        print(f"   {i+1}. {result.strip()[:100]}..." if len(result) > 100 else f"   {i+1}. {result.strip()}")

In [ ]:
# Compare Top-P settings  
prompt = "List 3 creative uses for a paperclip:"

print(f"Prompt: {prompt}\n")
print("=" * 60)

for top_p in [0.3, 0.6, 0.9]:
    print(f"\n🎯 Top-P = {top_p}:")
    result = generate_text(prompt, temperature=1.0, top_p=top_p)
    print(f"   {result.strip()}")

---

## 8. Choosing the Right Strategy

### Decision Guide

| Task | Temperature | Top-P | Top-K | Notes |
|------|-------------|-------|-------|-------|
| **Factual Q&A** | 0.0-0.3 | 0.1-0.3 | 1-5 | Deterministic, accurate |
| **Code generation** | 0.0-0.2 | 0.1-0.3 | 5-10 | Correctness matters |
| **Summarization** | 0.3-0.5 | 0.5-0.7 | 20-40 | Coherent but not rigid |
| **Chat/Dialogue** | 0.7-0.9 | 0.8-0.95 | 40-100 | Natural, varied |
| **Creative writing** | 0.9-1.2 | 0.9-0.95 | 100+ | Diverse, surprising |
| **Brainstorming** | 1.0-1.5 | 0.95-1.0 | All | Maximum diversity |

### Combining Parameters

```python
# Conservative (factual)
temperature=0.2, top_p=0.1

# Balanced (general use)
temperature=0.7, top_p=0.9

# Creative (stories, ideas)  
temperature=1.0, top_p=0.95
```

⚠️ **Note**: Don't combine high temperature with low top_p or vice versa - they can conflict!

In [ ]:
# Practical examples with recommended settings

tasks = {
    "Factual Q&A": {
        "prompt": "What is the atomic number of carbon?",
        "temperature": 0,
        "top_p": 0.1
    },
    "Code Generation": {
        "prompt": "Write a Python function to calculate factorial.",
        "temperature": 0.2,
        "top_p": 0.3
    },
    "Creative Writing": {
        "prompt": "Write a surprising plot twist for a mystery novel in one sentence.",
        "temperature": 1.0,
        "top_p": 0.95
    }
}

for task_name, config in tasks.items():
    print(f"\n{'='*60}")
    print(f"📋 Task: {task_name}")
    print(f"   Settings: temp={config['temperature']}, top_p={config['top_p']}")
    print(f"   Prompt: {config['prompt']}")
    print(f"\n   Response:")
    result = generate_text(config['prompt'], 
                          temperature=config['temperature'], 
                          top_p=config['top_p'])
    print(f"   {result.strip()}")

---

## 📝 Student Exercises

### Exercise 1: Find the Sweet Spot
Experiment with different temperature values to find the best setting for generating product descriptions.

In [ ]:
# Exercise 1: Experiment with product descriptions
product_prompt = "Write a compelling one-paragraph product description for wireless earbuds."

# TODO: Try different temperature values (0.3, 0.7, 1.0, 1.3)
# Which produces the best balance of creativity and professionalism?

# for temp in [0.3, 0.7, 1.0, 1.3]:
#     print(f"Temperature {temp}:")
#     print(generate_text(product_prompt, temperature=temp))
#     print()

### Exercise 2: Top-K vs Top-P
Compare the diversity of outputs using Top-K vs Top-P for the same prompt.

In [ ]:
# Exercise 2: Compare diversity
# Note: OpenAI API uses top_p (not top_k directly)
# But you can approximate: lower top_p ≈ lower top_k effect

creative_prompt = "Invent a new holiday and describe how people celebrate it."

# TODO: Generate 3 samples with top_p=0.5 and 3 samples with top_p=0.95
# Which produces more diverse outputs?

### Exercise 3: Task-Specific Settings
Design optimal sampling parameters for a customer support chatbot.

In [ ]:
# Exercise 3: Customer support chatbot
# Requirements:
# - Responses should be helpful and accurate
# - Some personality is okay, but not too creative
# - Should be consistent (low variance)

customer_query = "My order hasn't arrived yet. What should I do?"

# TODO: Choose temperature and top_p values and justify your choices
# your_temperature = ?
# your_top_p = ?

---

## 🎯 Key Takeaways

1. **LLMs generate text probabilistically** - sampling strategies control diversity

2. **Temperature** scales the probability distribution:
   - Low (0-0.3): Deterministic, focused
   - High (1.0+): Creative, diverse

3. **Top-K** limits to K most likely tokens (fixed cutoff)

4. **Top-P** limits to tokens covering P probability mass (adaptive cutoff)

5. **Match strategy to task**:
   - Factual → Low temperature
   - Creative → High temperature + high top_p

---

### Next: Fine-Tuning BERT Notebook →